In [ ]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime


def fetch_historical_data(symbol, timeframe, start_date, end_date):
    """
    Fetches historical price data for a single symbol and timeframe.

    Args:
        symbol (str): The trading symbol (e.g., "BTCUSD").
        timeframe (int): The timeframe for the bars (e.g., mt5.TIMEFRAME_H1).
        start_date (datetime): The start date for the data range.
        end_date (datetime): The end date for the data range.

    Returns:
        pd.DataFrame or None: DataFrame of historical price data, or None if
        the pull failed / returned no rows.
    """
    rates = mt5.copy_rates_range(symbol, timeframe, start_date, end_date)

    if rates is None or len(rates) == 0:
        print(f"[{symbol}] No data found or error: {mt5.last_error()}")
        return None

    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df['symbol'] = symbol

    print(f"[{symbol}] Successfully pulled {len(df)} bars "
          f"({df['time'].iloc[0]} -> {df['time'].iloc[-1]}).")

    return df


def fetch_multiple_symbols(symbols, timeframe, start_date, end_date,
                            terminal_path=None):
    """
    Fetches historical price data for multiple symbols over the same
    timeframe and date range, in a single MT5 session.

    Args:
        symbols (list[str]): Trading symbols, e.g. ["BTCUSD", "ETHUSD", "XAUUSD"].
        timeframe (int): mt5 timeframe constant (e.g. mt5.TIMEFRAME_H1).
        start_date (datetime): Start of the range.
        end_date (datetime): End of the range.
        terminal_path (str, optional): Path to terminal64.exe if you need to
            point at a specific broker terminal instead of the default one.

    Returns:
        dict[str, pd.DataFrame]: symbol -> DataFrame (only symbols that
        returned data are included).
    """
    init_ok = mt5.initialize(path=terminal_path) if terminal_path else mt5.initialize()
    if not init_ok:
        print(f"initialize() failed, error code: {mt5.last_error()}")
        return {}

    try:
        results = {}
        for symbol in symbols:
            # copy_rates_range needs the symbol to be selected/visible first
            if not mt5.symbol_select(symbol, True):
                print(f"[{symbol}] symbol_select failed, error code: {mt5.last_error()}")
                continue

            df = fetch_historical_data(symbol, timeframe, start_date, end_date)
            if df is not None:
                results[symbol] = df
    finally:
        # always shut down even if something above raises
        mt5.shutdown()

    return results


if __name__ == "__main__":
    # Example usage
    symbols = ["BTCUSD", "ETHUSD", "NVDA.US"]
    tf = mt5.TIMEFRAME_H1
    start = datetime(2026, 1, 1, 0, 0)
    end = datetime(2026, 6, 15, 0, 0)

    data = fetch_multiple_symbols(symbols, tf, start, end)

    for sym, df in data.items():
        print(f"\n{sym}:")
        print(df.head())

    # Optional: combine into one long-format DataFrame
    if data:
        combined = pd.concat(data.values(), ignore_index=True)
        output_file = "combined_historical_data.csv"
        combined.to_csv(output_file, index=False)
        print("\nCombined shape:", combined.shape)

In [ ]:
"""
Strategy framework.

Design:
- `Strategy` is an abstract base class. Every concrete strategy implements
  `generate_signals(data)` and returns a pd.Series of positions/signals
  indexed the same as the input data. Convention used here:
      1  -> long
     -1  -> short
      0  -> flat
  (Swap this convention for continuous position sizing if you prefer -
   just keep it consistent across strategies so they're comparable.)

- `Strategies` is a registry/orchestrator. It doesn't contain trading logic
  itself - it holds strategy instances, runs them, and lets you compare or
  combine their output. This keeps each strategy's logic isolated and
  independently testable.
"""

from abc import ABC, abstractmethod
import pandas as pd
import numpy as np


# ---------------------------------------------------------------------------
# Base interface
# ---------------------------------------------------------------------------

class Strategy(ABC):
    """Common interface every strategy must implement."""

    name: str = "base_strategy"

    @abstractmethod
    def generate_signals(self, data: pd.DataFrame) -> pd.Series:
        """
        Args:
            data: OHLCV DataFrame, indexed by time, with at least a 'close'
                  column (and 'symbol' if multi-symbol).

        Returns:
            pd.Series of {-1, 0, 1} aligned to data.index.
        """
        raise NotImplementedError


# ---------------------------------------------------------------------------
# Mean reversion
# ---------------------------------------------------------------------------

class MeanReversionStrategy(Strategy):
    """
    Z-score mean reversion on a rolling window: go long when price is
    `entry_z` std devs below its rolling mean, short when above, exit
    back toward `exit_z`.
    """

    name = "mean_reversion"

    def __init__(self, lookback: int = 20, entry_z: float = 2.0, exit_z: float = 0.5):
        self.lookback = lookback
        self.entry_z = entry_z
        self.exit_z = exit_z

    def generate_signals(self, data: pd.DataFrame) -> pd.Series:
        close = data['close']
        rolling_mean = close.rolling(self.lookback).mean()
        rolling_std = close.rolling(self.lookback).std()
        zscore = (close - rolling_mean) / rolling_std

        signals = pd.Series(0, index=data.index)
        signals[zscore < -self.entry_z] = 1    # oversold -> long
        signals[zscore > self.entry_z] = -1    # overbought -> short
        # flatten once it reverts back near the mean
        signals[zscore.abs() < self.exit_z] = 0
        signals = signals.replace(0, np.nan).ffill().fillna(0)

        return signals


# ---------------------------------------------------------------------------
# Statistical arbitrage variants
# ---------------------------------------------------------------------------

class StatArbCointegrationStrategy(Strategy):
    """
    Stat-arb #1: pairs trading via cointegrated spread z-score.
    Requires `data` to have two price columns (e.g. 'close_a', 'close_b')
    for the pair. Hedge ratio estimated via rolling OLS.
    """

    name = "stat_arb_cointegration"

    def __init__(self, col_a: str, col_b: str, lookback: int = 60,
                 entry_z: float = 2.0, exit_z: float = 0.5):
        self.col_a = col_a
        self.col_b = col_b
        self.lookback = lookback
        self.entry_z = entry_z
        self.exit_z = exit_z

    def generate_signals(self, data: pd.DataFrame) -> pd.Series:
        a, b = data[self.col_a], data[self.col_b]

        # rolling hedge ratio (beta) of a on b
        cov = a.rolling(self.lookback).cov(b)
        var_b = b.rolling(self.lookback).var()
        beta = cov / var_b

        spread = a - beta * b
        spread_mean = spread.rolling(self.lookback).mean()
        spread_std = spread.rolling(self.lookback).std()
        zscore = (spread - spread_mean) / spread_std

        signals = pd.Series(0, index=data.index)
        signals[zscore < -self.entry_z] = 1     # spread cheap -> long spread
        signals[zscore > self.entry_z] = -1     # spread rich -> short spread
        signals[zscore.abs() < self.exit_z] = 0
        signals = signals.replace(0, np.nan).ffill().fillna(0)

        return signals


class StatArbCrossSectionalStrategy(Strategy):
    """
    Stat-arb #2: cross-sectional mean reversion across a basket of symbols.
    Ranks symbols by short-term return, longs the worst performers and
    shorts the best (relative-value, not directional).

    `data` must be wide-format: one column per symbol of returns.
    """

    name = "stat_arb_cross_sectional"

    def __init__(self, lookback: int = 5, quantile: float = 0.2):
        self.lookback = lookback
        self.quantile = quantile  # fraction of universe to long/short

    def generate_signals(self, data: pd.DataFrame) -> pd.DataFrame:
        # returns data is wide: columns = symbols
        cum_ret = data.rolling(self.lookback).sum()

        signals = pd.DataFrame(0, index=data.index, columns=data.columns)
        for dt, row in cum_ret.iterrows():
            row = row.dropna()
            if len(row) < 5:
                continue
            n = max(1, int(len(row) * self.quantile))
            longs = row.nsmallest(n).index    # worst performers -> revert up
            shorts = row.nlargest(n).index    # best performers -> revert down
            signals.loc[dt, longs] = 1
            signals.loc[dt, shorts] = -1

        return signals


class StatArbKalmanStrategy(Strategy):
    """
    Stat-arb #3: pairs trading with a Kalman-filtered dynamic hedge ratio
    instead of a rolling-window OLS beta. Adapts faster to regime shifts
    than a fixed lookback.
    """

    name = "stat_arb_kalman"

    def __init__(self, col_a: str, col_b: str,
                 entry_z: float = 2.0, exit_z: float = 0.5,
                 delta: float = 1e-4, ve: float = 1e-3):
        self.col_a = col_a
        self.col_b = col_b
        self.entry_z = entry_z
        self.exit_z = exit_z
        self.delta = delta   # process noise (how fast beta can drift)
        self.ve = ve          # observation noise

    def generate_signals(self, data: pd.DataFrame) -> pd.Series:
        a, b = data[self.col_a].values, data[self.col_b].values
        n = len(a)

        beta = np.zeros(n)
        P = np.zeros(n)
        vw = self.delta / (1 - self.delta)

        beta[0] = 0.0
        P[0] = 1.0
        spread = np.zeros(n)

        for t in range(1, n):
            # predict
            beta_pred = beta[t - 1]
            P_pred = P[t - 1] + vw

            # update
            x = b[t]
            y = a[t]
            r = P_pred * x ** 2 + self.ve
            K = P_pred * x / r
            spread[t] = y - beta_pred * x
            beta[t] = beta_pred + K * spread[t]
            P[t] = (1 - K * x) * P_pred

        spread_series = pd.Series(spread, index=data.index)
        roll_mean = spread_series.rolling(30).mean()
        roll_std = spread_series.rolling(30).std()
        zscore = (spread_series - roll_mean) / roll_std

        signals = pd.Series(0, index=data.index)
        signals[zscore < -self.entry_z] = 1
        signals[zscore > self.entry_z] = -1
        signals[zscore.abs() < self.exit_z] = 0
        signals = signals.replace(0, np.nan).ffill().fillna(0)

        return signals


# ---------------------------------------------------------------------------
# ML-based
# ---------------------------------------------------------------------------

class MLStrategy(Strategy):
    """
    Wraps a trained sklearn/XGBoost-style classifier that predicts
    direction (or return sign) from a feature set. This class only
    handles inference + signal mapping; training/feature engineering
    should happen upstream and the fitted model passed in.
    """

    name = "ml"

    def __init__(self, model, feature_cols: list, threshold: float = 0.5,
                 long_class=1, short_class=0):
        """
        Args:
            model: fitted classifier exposing .predict_proba() or .predict()
            feature_cols: columns in `data` used as model input
            threshold: confidence threshold for predict_proba-based models
            long_class / short_class: which model output label maps to
                long / short (set short_class=None for a long-only model)
        """
        self.model = model
        self.feature_cols = feature_cols
        self.threshold = threshold
        self.long_class = long_class
        self.short_class = short_class

    def generate_signals(self, data: pd.DataFrame) -> pd.Series:
        X = data[self.feature_cols].dropna()
        signals = pd.Series(0, index=data.index)

        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X)
            classes = list(self.model.classes_)
            long_idx = classes.index(self.long_class)
            long_conf = proba[:, long_idx]

            preds = pd.Series(0, index=X.index)
            preds[long_conf >= self.threshold] = 1
            if self.short_class is not None:
                short_idx = classes.index(self.short_class)
                short_conf = proba[:, short_idx]
                preds[short_conf >= self.threshold] = -1
        else:
            raw = self.model.predict(X)
            preds = pd.Series(raw, index=X.index).map(
                lambda p: 1 if p == self.long_class
                else (-1 if p == self.short_class else 0)
            )

        signals.loc[preds.index] = preds
        return signals


# ---------------------------------------------------------------------------
# Registry / orchestrator
# ---------------------------------------------------------------------------

class Strategies:
    """
    Registry that holds strategy instances and runs them against data.
    Keeps individual strategy logic decoupled from execution/combination
    logic - add a new strategy by instantiating it and calling `register`.
    """

    def __init__(self):
        self._strategies: dict[str, Strategy] = {}

    def register(self, strategy: Strategy):
        self._strategies[strategy.name] = strategy
        return self  # allow chaining

    def run(self, name: str, data: pd.DataFrame):
        if name not in self._strategies:
            raise KeyError(f"No strategy registered under '{name}'. "
                            f"Available: {list(self._strategies)}")
        return self._strategies[name].generate_signals(data)

    def run_all(self, data: pd.DataFrame) -> pd.DataFrame:
        """Run every registered strategy on the same data and return a
        DataFrame of signals, one column per strategy."""
        out = {}
        for name, strat in self._strategies.items():
            try:
                out[name] = strat.generate_signals(data)
            except Exception as e:
                print(f"[{name}] failed: {e}")
        return pd.DataFrame(out)

    def combine(self, data: pd.DataFrame, weights: dict = None) -> pd.Series:
        """
        Simple ensemble: weighted average of each strategy's signal,
        then sign() to get a final discrete position. Equal-weighted
        by default.
        """
        signals = self.run_all(data)
        if weights is None:
            weights = {name: 1.0 for name in signals.columns}

        weighted = sum(signals[name] * w for name, w in weights.items()
                        if name in signals.columns)
        return np.sign(weighted)

    def __repr__(self):
        return f"Strategies({list(self._strategies)})"


# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # dummy price data

    

    strategies = Strategies()
    strategies.register(MeanReversionStrategy(lookback=20, entry_z=2.0))
    # register stat-arb / ML strategies similarly once you have paired
    # data or a fitted model, e.g.:
    # strategies.register(StatArbCointegrationStrategy("close_a", "close_b"))
    # strategies.register(MLStrategy(model=my_fitted_model, feature_cols=[...]))

    print(strategies)
    signals = strategies.run("mean_reversion", df)
    print(signals.tail())